# Level 1 — Single-Nucleus Census of an Adult Glioblastoma

## CAJAL "Neuromics 2026" — Computational Mini-Project C10 (Level 1)

**Estimated time:** ~2 days

**Learning objectives**
- Load and explore single-nucleus RNA-seq (snRNA-seq) data in the AnnData format
- Perform quality control on nuclei and understand what has *already* been done to the data
- Normalize, select features, and integrate across donors (Harmony **and** scVI, compared)
- Cluster and annotate broad cell types using marker genes **and** an automated classifier
- Separate malignant cells from the non-malignant tumour microenvironment (TME) using copy-number inference
- Characterise a continuous **malignant cell-state axis** and how it varies between donors

**Dataset:** snRNA-seq (10x Multiome, RNA modality) from **two adult patients** with high-grade glioma — donors `AT10` and `AT14`, sampled from several tissue sites each. `~118,000` nuclei × `~36,600` genes. The expression matrix holds **raw integer UMI counts**.

> **We are not telling you where this data comes from yet.** There is no paper, title, or figure to look up. Your job is to let the data speak: reconstruct the cell types and tumour states yourself, the way the original analysts had to. The "reveal" happens in Level 2.

---

## 0. Setup

In [ ]:
# Your code for: this step


## 1. Load and explore the data

🔬 **TASK 1.1:** Load the snRNA-seq dataset and inspect the AnnData object — shape, `.obs` columns, and confirm what is stored in `.X`.

In [ ]:
# Your code for: Load the snRNA-seq dataset and inspect the AnnData object — shape, `.obs` columns, and confirm what is stored in `.X`


In [ ]:
# Your code for: Load the snRNA-seq dataset and inspect the AnnData object — shape, `.obs` columns, and confirm what is stored in `.X`


🔬 **TASK 1.2:** Confirm that `.X` holds raw integer counts — every downstream choice (normalization, `seurat_v3` HVGs, scVI, CNV inference) depends on this.

In [ ]:
# Your code for: Confirm that `.X` holds raw integer counts — every downstream choice (normalization, `seurat_v3` HVGs, scVI, CNV inference) depends on this


❓ **QUESTION:** The two donors contribute very different nuclei counts (`AT10` ≈ 86k vs `AT14` ≈ 32k), and each donor was sampled from several physical tissue sites. Why might naively pooling all nuclei and clustering immediately be a problem — and which `.obs` column is the natural *batch* variable to correct for later?

## 2. Quality control

Not every nuclear barcode is a good nucleus: some are empty/ambient, some are doublets (two nuclei in one droplet), some are stressed. The three workhorse single-cell QC metrics are:

| Metric | Low values suggest | High values suggest |
|---|---|---|
| total counts (UMIs) per nucleus | empty droplet / debris | doublet |
| genes detected per nucleus | low-complexity / empty | doublet |
| % mitochondrial counts | (in nuclei, mito is expected *low*) | cytoplasmic leakage / stress |

In human data, mitochondrial genes start with `MT-`. We also have a precomputed Scrublet **doublet score** per nucleus in `.obs['doublet_scores']`.

🔬 **TASK 2.1:** Flag mitochondrial genes and compute QC metrics with `sc.pp.calculate_qc_metrics`.

In [ ]:
# Your code for: Flag mitochondrial genes and compute QC metrics with `sc.pp.calculate_qc_metrics`


🔬 **TASK 2.2:** Visualise the QC distributions, split by donor, and look at the joint counts-vs-genes structure.

In [ ]:
# Your code for: Visualise the QC distributions, split by donor, and look at the joint counts-vs-genes structure


In [ ]:
# Your code for: Visualise the QC distributions, split by donor, and look at the joint counts-vs-genes structure


🔬 **TASK 2.3:** Look at the *floors* of these distributions before choosing thresholds. A standard nuclei-QC recipe removes nuclei with `< ~500` genes, `< ~1000` UMIs, or `> 10%` mito. Check how many nuclei those cuts would actually remove here.

In [ ]:
# Your code for: Look at the *floors* of these distributions before choosing thresholds. A standard nuclei-QC recipe removes nuclei with `< ~500` genes, `< ~1000` UMIs, or `> 10%` mito. Check how many nuclei those cuts would actually remove here


💡 **HINT:** Notice the floors — `total_counts` bottoms out right at ~1000, `n_genes_by_counts` at ~500, and `pct_counts_mt` never exceeds ~10%. That is not a coincidence: **basic per-nucleus QC has already been applied to this dataset**, so the genes/UMI/mito filters remove essentially nothing. That is realistic — you are often handed data that has had a first QC pass. The QC lever that is still meaningful here is **doublet removal**, which has *not* been applied.

🔬 **TASK 2.4:** Apply QC. Keep the standard cuts as explicit guardrails (so the notebook is correct even on un-filtered data), and add a Scrublet doublet cut at `0.25` (the upper tail in the histogram above). Then drop genes seen in `< 3` nuclei. Report nuclei remaining.

In [ ]:
# Your code for: Apply QC. Keep the standard cuts as explicit guardrails (so the notebook is correct even on un-filtered data), and add a Scrublet doublet cut at `0.25` (the upper tail in the histogram above). Then drop genes seen in `< 3` nuclei. Report nuclei remaining


⚠️ **CHECKPOINT:** You should have roughly **116,000–118,000 nuclei** remaining (the doublet cut removes ~1,000–1,500; the genes/UMI/mito cuts remove ~0 because the data was pre-filtered). After the `min_cells=3` gene filter you should keep on the order of **33,000–34,000 genes**. If you lost tens of thousands of nuclei, your thresholds are far too strict for *this* dataset — re-read the floors above.

## 3. Normalization and feature selection

Raw counts conflate biology with sequencing depth. We library-size-normalize, log-transform, snapshot the result into `.raw` (for marker plotting later), then select highly variable genes (HVGs).

🔬 **TASK 3.1:** Stash raw counts in a layer, then `normalize_total` (target 1e4) + `log1p`, and freeze `.raw`.

In [ ]:
# Your code for: Stash raw counts in a layer, then `normalize_total` (target 1e4) + `log1p`, and freeze `.raw`


🔬 **TASK 3.2:** Select ~3000 HVGs with `flavor="seurat_v3"` on the **raw counts layer** (this flavor models mean–variance on counts, not log data), computed per-donor (`batch_key`) so donor-specific genes don't dominate.

In [ ]:
# Your code for: Select ~3000 HVGs with `flavor="seurat_v3"` on the **raw counts layer** (this flavor models mean–variance on counts, not log data), computed per-donor (`batch_key`) so donor-specific genes don't dominate


❓ **QUESTION:** Why do we run `seurat_v3` HVG selection on the *raw counts* layer rather than on the log-normalized matrix? And why pass `batch_key="donor_id"` — what failure mode does that guard against?

## 4. Dimensionality reduction and **donor integration**

We have two donors. If donor identity drives the top axes of variation, clusters will split by patient instead of by biology. We will:
1. scale HVGs → PCA (the uncorrected embedding),
2. run **two** integration methods over PCA/counts keyed on `donor_id` — **Harmony** and **scVI** — and compare them,
3. pick one to carry downstream via an `INTEGRATION_METHOD` flag, keeping *both* embeddings in `.obsm` so the flag works either way.

🔬 **TASK 4.1:** Scale HVGs, run PCA (50 comps), inspect the variance elbow.

In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


💡 **HINT:** The elbow flattens by ~30 PCs; we use 30 for neighbor graphs throughout. First let's *see* the donor (batch) effect in the uncorrected space.

In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


🔬 **TASK 4.2 — Harmony.** Integrate the PCA embedding on `donor_id`. (We call `harmonypy` directly and reshape its output ourselves: the installed `harmonypy`/`scanpy` combination returns `Z_corr` in a shape that the thin `sc.external.pp.harmony_integrate` wrapper mishandles — a good reminder that library version mismatches are a normal part of real analysis.)

In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


🔬 **TASK 4.3 — scVI.** scVI is a deep generative model trained on raw counts. On CPU it is far slower than Harmony, so **probe per-epoch cost first**, compare against scvi-tools' own epoch heuristic, and cap `max_epochs` so this stays a teaching-friendly runtime (target: well under an hour on CPU).

In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


💡 **HINT — runtime caveat.** scVI training cost scales with dataset size and is genuinely slow on CPU for tens of thousands of cells — avoid scvi-tools' automatic epoch heuristic (`get_max_epochs_heuristic`) for a teaching session; in this environment it picked an epoch count that made the full run take well over an hour with no early feedback, which is the wrong trade for a live class. Use a small, **fixed** epoch count instead — you always know your runtime budget up front. Raise it if you have time or GPU access.

In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


🔬 **TASK 4.4 — compare integrations.** Build a neighbor graph + UMAP for each embedding and inspect donor mixing visually. Then quantify mixing with a simple **neighbor-purity** proxy: for each nucleus, the fraction of its k nearest neighbors from the *same* donor. Under perfect mixing this approaches each donor's global fraction; under no correction it approaches 1.0.

In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


In [ ]:
# Your code for: Scale HVGs, run PCA (50 comps), inspect the variance elbow


💡 **HINT:** Both methods pull the mean same-donor neighbor fraction down from the uncorrected value toward the perfect-mixing baseline. For this two-donor teaching run we proceed with **Harmony**: it integrated in seconds (vs minutes for scVI on CPU), gives comparable mixing, and operates on the same interpretable PCA space. We keep `X_scvi` in `.obsm` so you can flip the flag and reproduce everything downstream with scVI instead.

🔬 **TASK 4.5:** Set the integration flag and build the **canonical** neighbor graph + UMAP that all downstream steps use.

In [ ]:
# Your code for: Set the integration flag and build the **canonical** neighbor graph + UMAP that all downstream steps use


## 5. Clustering

Leiden community detection on the integrated neighbor graph. We sweep a few resolutions and pick one whose granularity is sensible for broad cell typing.

🔬 **TASK 5.1:** Run Leiden at resolutions 0.3 / 0.5 / 1.0 and compare cluster counts and UMAPs.

In [ ]:
# Your code for: Run Leiden at resolutions 0.3 / 0.5 / 1.0 and compare cluster counts and UMAPs


💡 **HINT:** We adopt **resolution 1.0** as the working clustering. Broad cell typing plus a malignant-state axis needs enough granularity to separate (a) the major TME lineages from malignant cells and (b) distinct malignant programs, without shattering into dozens of micro-clusters. Lower resolutions merged biologically distinct populations in this dataset.

In [ ]:
# Your code for: Run Leiden at resolutions 0.3 / 0.5 / 1.0 and compare cluster counts and UMAPs


🔬 **TASK 5.2:** Check each cluster's donor composition — this distinguishes shared biology (both donors) from donor-private populations.

In [ ]:
# Your code for: Check each cluster's donor composition — this distinguishes shared biology (both donors) from donor-private populations


## 6. Broad cell-type annotation

We assign each cluster a coarse cell-type label using **two independent lines of evidence**, then reconcile them:
1. **Marker genes** — a curated TME/lineage panel (`gbmspace_utils.TME_MARKERS`), via dotplot and per-cluster gene scores.
2. **CellTypist** — an automated reference classifier; we use the `Developing_Human_Brain` model (a developmental-brain reference, appropriate for glioma whose malignant cells mimic neurodevelopmental programs).

🔬 **TASK 6.1:** Rank marker genes per cluster (Wilcoxon) for a first look at what defines each cluster.

In [ ]:
# Your code for: Rank marker genes per cluster (Wilcoxon) for a first look at what defines each cluster


🔬 **TASK 6.2:** Dotplot the curated TME/lineage marker panel across clusters.

In [ ]:
# Your code for: Dotplot the curated TME/lineage marker panel across clusters


In [ ]:
# Your code for: Dotplot the curated TME/lineage marker panel across clusters


🔬 **TASK 6.3:** Run **CellTypist** with the `Developing_Human_Brain` model on the log-normalized data, then look at the dominant CellTypist call per Leiden cluster.

In [ ]:
# Your code for: Run **CellTypist** with the `Developing_Human_Brain` model on the log-normalized data, then look at the dominant CellTypist call per Leiden cluster


🔬 **TASK 6.4:** Reconcile marker evidence + CellTypist into one coarse `cell_type` per cluster. The cell below prints, for every cluster, its top DE genes, its strongest TME signature, and its CellTypist call side-by-side — the inputs to a manual majority decision.

In [ ]:
# Your code for: Reconcile marker evidence + CellTypist into one coarse `cell_type` per cluster. The cell below prints, for every cluster, its top DE genes, its strongest TME signature, and its CellTypist call side-by-side — the inputs to a manual majority decision


💡 The reconciliation logic (filled in after reading the table above): clusters whose top markers and TME signature are **microglia** (`P2RY12`, `CX3CR1`), **macrophage/myeloid** (`CD163`, `CD14`), **oligodendrocyte** (`PLP1`, `MOBP`, `ST18`), **neuronal** (`RBFOX3`, `GAD1/2`, `SLC17A7`), **lymphocyte** (`CD3E`, `CD2`), or **vascular** (`CLDN5`, `PDGFRB`) are clearly non-malignant TME. The remaining large clusters express glial/progenitor programs (`SLC1A3`, `GFAP`, `EGFR`, `PDGFRA`, `OLIG1/2`) and CellTypist calls them glioblast / radial-glia / OPC — these are the candidate **malignant** populations (to be confirmed by copy-number inference in Section 7). The dictionary below encodes that per-cluster decision.

🔬 **TASK 6.5:** Encode the per-cluster label decision and write the `cell_type` column.

In [ ]:
# Your code for: Encode the per-cluster label decision and write the `cell_type` column


In [ ]:
# Your code for: Encode the per-cluster label decision and write the `cell_type` column


❓ **QUESTION:** Where did marker-based and CellTypist calls *disagree*, and how did you break the tie? CellTypist was trained on developing (normal) brain — why is it expected to label malignant glioma cells as "glioblast" / "radial glia" / "OPC", and why is that *useful* rather than a failure?

## 7. Malignant vs TME — copy-number inference

Glioma cells carry large-scale **copy-number alterations (CNAs)** — whole-arm gains/losses absent from normal brain cells. We infer per-nucleus CNA profiles from expression with **infercnvpy**, using clearly-normal TME clusters as the diploid **reference**. Malignant cells show stronger genome-wide CNA signal *and* higher correlation to the average malignant profile.

🔬 **TASK 7.1:** Attach gene chromosomal positions (GRCh38) to `.var` — infercnvpy needs `chromosome`, `start`, `end` to order genes along the genome.

In [ ]:
# Your code for: Attach gene chromosomal positions (GRCh38) to `.var` — infercnvpy needs `chromosome`, `start`, `end` to order genes along the genome


🔬 **TASK 7.2:** Pick the unambiguous TME clusters as the CNA reference and run `infercnvpy`.

In [ ]:
# Your code for: Pick the unambiguous TME clusters as the CNA reference and run `infercnvpy`


🔬 **TASK 7.3:** Build a per-nucleus **CNA correlation** = correlation of each nucleus's CNA profile to the mean profile of high-CNA nuclei (a malignant-consensus profile). Malignant cells score high on *both* CNA signal and CNA correlation.

In [ ]:
# Your code for: Build a per-nucleus **CNA correlation** = correlation of each nucleus's CNA profile to the mean profile of high-CNA nuclei (a malignant-consensus profile). Malignant cells score high on *both* CNA signal and CNA correlation


💡 **HINT — thresholds, and an honest caveat about the source.** A common cell-level malignant call combines a CNA-signal cut with a CNA-correlation cut (order-of-magnitude: signal `> ~0.02`, correlation `> ~0.3`). For the **cluster-level** "is this cluster malignant" call, the standard is a fraction-of-malignant-cells cut (we use `> 20%`).

⚠️ Worth knowing for later: published CNA pipelines are not always internally consistent. The kind of methods text these thresholds come from will sometimes state one cluster-level cutoff (e.g. "≥ 3% malignant") while the *figure legend* of the same work states another (e.g. "≥ 5%"). We flag this not to be pedantic but because **real papers contain exactly these small inconsistencies** — part of learning to read them critically is noticing when the methods and the figures disagree, and deciding for yourself.

🔬 **TASK 7.4:** Classify nuclei (cell-level) and clusters (cluster-level), and set a derived `cell_status`.

In [ ]:
# Your code for: Classify nuclei (cell-level) and clusters (cluster-level), and set a derived `cell_status`


In [ ]:
# Your code for: Classify nuclei (cell-level) and clusters (cluster-level), and set a derived `cell_status`


⚠️ **CHECKPOINT:** With two adult glioma donors you should land on a **malignant majority** — very roughly **55–65% of nuclei malignant**, the rest TME (dominated by oligodendrocytes and myeloid cells). If you got almost everything malignant or almost nothing, re-check (a) your reference clusters in 7.2 and (b) that gene positions attached in 7.1.

## 8. The malignant cell-state axis

This is the conceptual core. Malignant glioma cells are not a set of discrete types but occupy a **continuum of states** mimicking neurodevelopmental and reactive-glial programs. We score each malignant nucleus against a panel of state signatures (`MALIGNANT_AXIS_MARKERS`), assign each its dominant state, then collapse to four major classes via `MAJOR_CLASS_OF`.

🔬 **TASK 8.1:** Subset to malignant nuclei and score the state signatures with the project helper `score_axis()`.

In [ ]:
# Your code for: Subset to malignant nuclei and score the state signatures with the project helper `score_axis()`


🔬 **TASK 8.2:** Assign each nucleus its dominant state (`assign_dominant_state`), then map to the 4 major classes.

In [ ]:
# Your code for: Assign each nucleus its dominant state (`assign_dominant_state`), then map to the 4 major classes


In [ ]:
# Your code for: Assign each nucleus its dominant state (`assign_dominant_state`), then map to the 4 major classes


🔬 **TASK 8.3:** Dotplot the state-signature scores per dominant state to confirm each state is driven by its own markers.

In [ ]:
# Your code for: Dotplot the state-signature scores per dominant state to confirm each state is driven by its own markers


🔬 **TASK 8.4 (bonus — myth-busting).** "Mesenchymal-like" glioma states are sometimes framed as classical EMT. Check whether canonical **EMT regulators** (`EMT_MARKERS`: `SNAI1/2`, `TWIST1/2`, `ZEB1/2`) are actually specific to the gliosis/hypoxia states here.

In [ ]:
# Your code for: Dotplot the state-signature scores per dominant state to confirm each state is driven by its own markers


❓ **QUESTION:** Are the classical EMT regulators strongly and *specifically* up in the gliosis/hypoxia (AC-gliosis-hypoxia) class, or are they weak/diffuse across all states? What does that imply about calling these states "mesenchymal" in the EMT sense?

## 9. Composition and differential expression

🔬 **TASK 9.1:** Write the malignant-state labels back onto the full object and compare cell-type and malignant-class composition between the two donors.

In [ ]:
# Your code for: Write the malignant-state labels back onto the full object and compare cell-type and malignant-class composition between the two donors


🔬 **TASK 9.2:** Pick one DE comparison. We contrast the two largest malignant major classes (Wilcoxon) and report top markers + a volcano.

In [ ]:
# Your code for: Pick one DE comparison. We contrast the two largest malignant major classes (Wilcoxon) and report top markers + a volcano


In [ ]:
# Your code for: Pick one DE comparison. We contrast the two largest malignant major classes (Wilcoxon) and report top markers + a volcano


## 10. Publication-quality summary figure

🔬 **TASK 10.1:** Assemble a single multi-panel figure synthesizing the analysis: the cell-type map, the malignant/TME split, and the malignant-state axis.

In [ ]:
# Your code for: Assemble a single multi-panel figure synthesizing the analysis: the cell-type map, the malignant/TME split, and the malignant-state axis


## 11. Save the annotated reference

This annotated object becomes the **reference for Level 2** (spatial deconvolution). We save it with clean, well-named `.obs` columns and both integration embeddings retained.

🔬 **TASK 11.1:** Restore full genes, tidy `.obs`, and write the annotated AnnData to `data/processed/`.

In [ ]:
# Your code for: Restore full genes, tidy `.obs`, and write the annotated AnnData to `data/processed/`


## Summary

You have, from raw counts and no labels:
1. ✅ QC'd the nuclei (discovering basic QC was pre-applied; added doublet removal)
2. ✅ Normalized, selected HVGs, and **integrated two donors** with Harmony and scVI (compared)
3. ✅ Clustered and annotated **broad cell types** (markers + CellTypist, reconciled)
4. ✅ Split **malignant vs TME** by copy-number inference
5. ✅ Characterised a continuous **malignant cell-state axis** and its per-donor composition
6. ✅ Run a DE comparison and built a publication figure
7. ✅ Saved a clean annotated reference for Level 2

**Carry into Level 2:** which malignant states did you find, and how were they distributed between donors? Level 2 asks *where* in the tissue these states live — and is where the dataset's origin is finally revealed.

---